# Qwen2.5-1.5B QLoRA — Vietnamese Dialogue Rewriter (Kaggle)

Notebook chạy end-to-end trên Kaggle GPU **T4 16GB**.

**Settings (panel bên phải):**
- Accelerator: `GPU T4 x1` (hoặc x2)
- Internet: **On** (bắt buộc để `pip install` và tải Qwen từ HuggingFace)

Trước khi Run All, sửa biến `REPO_URL` ở cell tiếp theo cho đúng repo của bạn.

## 1. Lấy code

Chọn một trong hai cách bằng cách bật `USE_KAGGLE_DATASET`.

- `False` → clone từ GitHub (`REPO_URL`).
- `True` → copy từ Kaggle Dataset đã upload (`KAGGLE_DATASET_DIR`).

In [ ]:
import os, shutil, subprocess

USE_KAGGLE_DATASET = False
REPO_URL = "https://github.com/thao12345310/qwen-lora-finetuning.git"  # sửa lại
KAGGLE_DATASET_DIR = "/kaggle/input/qwen-lora-finetuning"           # sửa lại nếu khác
PROJECT_DIR = "/kaggle/working/qwen-lora-finetuning"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

if USE_KAGGLE_DATASET:
    shutil.copytree(KAGGLE_DATASET_DIR, PROJECT_DIR)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print("CWD:", os.getcwd())
print(os.listdir())

## 2. Cài dependencies

`requirements.txt` (data/serving utils) + **Llama Factory** (engine fine-tune). LlamaFactory kéo theo transformers/peft/accelerate tương thích.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "llamafactory[torch,bitsandbytes]"
import llamafactory, subprocess
print("llamafactory", llamafactory.__version__)
!llamafactory-cli version

## 3. (Tuỳ chọn) HuggingFace token

Bỏ qua nếu không bị rate-limit. Để dùng: Add-ons → Secrets → thêm secret `HF_TOKEN`.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("Skip HF token:", e)

## 4. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 5. Chia train/valid/test từ dataset vLLM đã gen sẵn

Dùng `data/train/dialogues_train_vllm.jsonl` (Phase-2 vLLM, format `conversations` chuẩn) — **không** sinh lại data. `split_data.py` stratify theo `domain × context_required` nên giữ đúng tỷ lệ 2/3 retrieve : 1/3 no-retrieve.

In [ ]:
# Dùng dataset vLLM đã gen sẵn (KHÔNG sinh lại data).
# split_data.py stratify theo domain × context_required -> giữ tỷ lệ 2/3 retrieve : 1/3 no-retrieve.
TRAIN_FILE = "data/train/dialogues_train_vllm.jsonl"
assert os.path.isfile(TRAIN_FILE), (
    f"Không thấy {TRAIN_FILE}. Hãy commit & push file này lên GitHub trước khi clone trên Kaggle."
)
!python src/data/split_data.py --input {TRAIN_FILE}
!ls -la data/processed/

## 6. Fine-tune QLoRA (Llama Factory)

Engine: `llamafactory-cli train` với `configs/qwen_lora_sft_llamafactory.yaml` (Qwen2.5-1.5B, LoRA r=8, NF4 4-bit, template qwen, fp16 cho T4).

- **Auto-resume:** `overwrite_output_dir: false` -> nếu phiên Kaggle ngắt, Run lại cell là học tiếp từ checkpoint mới nhất.
- **Đa GPU:** T4 ×2 -> cell tự đặt `FORCE_TORCHRUN=1` để LlamaFactory chạy **DDP** trên cả 2 GPU (~gấp đôi tốc độ). Effective batch khi đó = `n_gpu × batch × grad_accum` = 2×1×8 = **16**; muốn giữ 8 thì sửa `gradient_accumulation_steps: 4` trong config.
- **Push HF:** đặt `PUSH_TO_HUB = True` (cần secret `HF_TOKEN` quyền write ở mục 3, sửa `HUB_MODEL_ID`).

Nếu OOM: giảm `cutoff_len: 512` hoặc `lora_rank: 4` trong config.

In [ ]:
import os, yaml, torch

PUSH_TO_HUB  = False  # True = đẩy adapter lên HuggingFace Hub sau khi train
HUB_MODEL_ID = "ThaoDuongDoingStuff/qwen-vi-rewriter-lora"  # sửa thành <username>/<repo> của bạn

# Đọc config repo, inject toggle push HF -> ghi ra config runtime.
cfg = yaml.safe_load(open("configs/qwen_lora_sft_llamafactory.yaml"))
if PUSH_TO_HUB:
    assert os.environ.get("HF_TOKEN"), "PUSH_TO_HUB cần secret HF_TOKEN (quyền write) — chạy cell mục 3 trước."
    cfg["push_to_hub"] = True
    cfg["hub_model_id"] = HUB_MODEL_ID
RUN_CFG = "/kaggle/working/lf_run.yaml"
yaml.safe_dump(cfg, open(RUN_CFG, "w"), allow_unicode=True, sort_keys=False)

# FORCE_TORCHRUN=1 -> LlamaFactory chạy DDP trên tất cả GPU khi có >1.
n_gpu = torch.cuda.device_count()
env = "FORCE_TORCHRUN=1 " if n_gpu > 1 else ""
cmd = f"{env}llamafactory-cli train {RUN_CFG}"
print(f"GPUs: {n_gpu} | auto-resume nếu có checkpoint\n{cmd}")
get_ipython().system(cmd)

## 7. So sánh base model vs fine-tuned trên test set

In [ ]:
!python -m src.inference.compare

In [ ]:
import json

with open("outputs/eval_results/comparison.jsonl", encoding="utf-8") as f:
    rows = [json.loads(l) for l in f]

print(f"Total: {len(rows)} samples\n")
for r in rows[:5]:
    print("DIALOGUE:", r["dialogue"])
    print("GOLD    :", r["gold"])
    print("BASE    :", r["base_output"])
    print("LORA    :", r["finetuned_output"])
    print("-" * 80)

## 8. Inference thử một hội thoại

In [ ]:
!python -m src.inference.predict \
  --conversation '[{"role":"user","content":"mở điều hoà"},{"role":"bot","content":"bạn muốn đặt bao nhiêu độ?"},{"role":"user","content":"27 độ"}]'

## 9. Lưu adapter ra `/kaggle/working` để download

Sau khi *Save Version* (Commit), mọi file trong `/kaggle/working` sẽ hiện trong tab Output của notebook.

In [ ]:
import shutil

src = "outputs/qwen-dialogue-rewriter-lora"
dst = "/kaggle/working/qwen-dialogue-rewriter-lora"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

# Zip cho dễ download
shutil.make_archive("/kaggle/working/qwen-dialogue-rewriter-lora", "zip", src)
print("Saved adapter + zip to /kaggle/working/")
!ls -lh /kaggle/working/